# Clustering from first principles: retail customers and business action

A retailer wants differentiated lifecycle strategies, but the data contain no correct segment label. Clustering will always return groups. The task is to decide whether those groups are stable, distinct, understandable, and useful enough to guide an experiment.

This workbook follows the central ideas emphasized in ISLP, Stanford CS229, MIT 15.097, and MIT CSAIL 6.S079: similarity is a modeling choice, the mean minimizes squared-distance cost, alternating k-means updates decrease that objective, initialization matters, internal scores do not prove meaning, and different algorithms encode different cluster shapes.

## Where clustering sits in machine learning

The first question is whether each observation comes with an outcome label.

- **Supervised learning:** an outcome is observed during training. Classification predicts a category, while regression predicts a quantity. Logistic regression, trees, and SVMs belong here because historical examples provide the answer the model should learn to predict.
- **Unsupervised learning:** no outcome label identifies the correct answer. Clustering looks for grouping structure, while methods such as PCA look for a lower-dimensional representation.

For the retail case, the transaction table contains customer behavior but no column named `correct_segment`. K-means therefore optimizes a geometric objective rather than prediction error against known labels. Business evidence must supply the meaning after the grouping is created.

**Connection to the previous topic:** SVM asks, ‘Given labeled examples of 3s and 8s, where should the decision boundary go?’ Clustering asks, ‘Given customer profiles without segment labels, what definition of similarity produces useful groups?’

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

candidates = [
    Path('../../modern-ai-ml-exercises/exercise-04-segment-studio/data/customers.csv'),
    Path('../modern-ai-ml-exercises/exercise-04-segment-studio/data/customers.csv'),
    Path('modern-ai-ml-exercises/exercise-04-segment-studio/data/customers.csv'),
    Path('/content/customers.csv'),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place customers.csv beside the notebook or run from the course workspace.')
customers = pd.read_csv(DATA_PATH)
print('loaded', DATA_PATH, customers.shape)

## 1. Start with the decision, not the algorithm

**Unit:** one customer profile derived from transaction history.

**Possible action:** choose which lifecycle message or service treatment to test.

**Failure condition:** clusters differ mathematically but support no feasible difference in action.

Before running a model, write one sentence describing what marketing would do differently for two credible segments.

In [ ]:
features = [
    'recency_days', 'purchase_frequency', 'monetary_value',
    'product_diversity', 'average_basket_value', 'return_line_rate'
]
customers[features].describe(percentiles=[.1,.5,.9,.99]).T.round(2)

## 2. Similarity is a design choice

Euclidean distance gives greater influence to features with larger numerical ranges. Spending is also highly skewed, so a few large customers can pull centroids. We will compare raw data with a log transform and standardization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].hist(customers.monetary_value, bins=50, color='#2457A6')
axes[0].set_title('Raw annual value: a long right tail')
axes[0].set_xlabel('monetary value')
axes[1].hist(np.log1p(customers.monetary_value), bins=50, color='#0F8A8A')
axes[1].set_title('log(1+x): large values remain large, but less dominant')
axes[1].set_xlabel('log monetary value')
plt.tight_layout(); plt.show()

In [ ]:
X_raw = customers[features].clip(lower=0).copy()
X_log = np.log1p(X_raw)
scaler = StandardScaler()
Z = scaler.fit_transform(X_log)

audit = pd.DataFrame({
    'raw_range': X_raw.max()-X_raw.min(),
    'raw_std': X_raw.std(),
    'standardized_std': pd.DataFrame(Z, columns=features).std()
})
audit.round(2)

**Interpretation check:** Standardization does not make features equally important in a business sense. It prevents units alone from deciding importance. Feature inclusion still requires judgment.

## 3. One k-means iteration by hand

K-means repeats two moves: assign each customer to the nearest centroid, then move each centroid to the mean of its assigned customers. The objective is the sum of squared distances to assigned centroids.

In [ ]:
# A tiny two-feature example, taken from real customer rows after preparation.
tiny = pd.DataFrame(Z[:, [0,2]], columns=['recency_scaled','value_scaled']).iloc[[5,25,80,120,300,500]].copy()
centroids_before = tiny.iloc[[0,4]].to_numpy()
distances = np.sqrt(((tiny.to_numpy()[:,None,:]-centroids_before[None,:,:])**2).sum(axis=2))
assignment = distances.argmin(axis=1)
centroids_after = np.vstack([tiny.to_numpy()[assignment==j].mean(axis=0) for j in [0,1]])
worked = tiny.copy()
worked['distance_to_A'] = distances[:,0]
worked['distance_to_B'] = distances[:,1]
worked['assigned_cluster'] = assignment
print('centroids before:\n', np.round(centroids_before,2))
print('centroids after one update:\n', np.round(centroids_after,2))
worked.round(2)

### Why the centroid update uses the mean

For one fixed cluster `C`, the k-means cost is `sum_{i in C} ||x_i - z||^2`. The mean minimizes this squared-distance cost. MIT's clustering notes express the result as a decomposition: moving the representative away from the mean adds `|C| ||z - mean(C)||^2` to the cost.

This gives k-means a rigorous two-move interpretation:

1. With centroids fixed, assigning every point to its nearest centroid cannot increase the objective.
2. With assignments fixed, replacing each centroid by its cluster mean cannot increase the objective.

The objective therefore decreases until the algorithm stops, but it can stop at a local rather than global optimum. Multiple initializations address that risk.

In [ ]:
# One random start can settle at a different local solution.
restart_results = []
for seed in range(12):
    fit = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed).fit(Z)
    restart_results.append({'seed': seed, 'final_inertia': fit.inertia_, 'iterations': fit.n_iter_})
pd.DataFrame(restart_results).sort_values('final_inertia').round(1)

### Nearest-centroid assignment creates Voronoi cells

Each centroid owns the region of feature space closer to it than to any other centroid. These nearest-centroid regions are Voronoi cells with straight boundaries. This explains the geometric bias of k-means toward compact, roughly convex groups. Curved or density-connected groups can be split even when they look coherent to a person.

## 4. K-means solves its objective, not the business problem

Inertia always decreases as `k` grows. Silhouette balances cohesion and separation. Stability asks whether repeated plausible fits produce similar assignments. None of these measures proves that a marketing action will work.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(Z), min(1500, len(Z)), replace=False)
rows = []
labels_by_k = {}
for k in range(2, 9):
    seed_labels = []
    for seed in range(5):
        seed_labels.append(KMeans(n_clusters=k, n_init=20, random_state=seed).fit_predict(Z))
    labels_by_k[k] = seed_labels[0]
    stability = np.mean([
        adjusted_rand_score(seed_labels[i], seed_labels[j])
        for i in range(5) for j in range(i+1,5)
    ])
    fitted = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE).fit(Z)
    rows.append({
        'k': k,
        'inertia': fitted.inertia_,
        'silhouette': silhouette_score(Z[sample_idx], fitted.labels_[sample_idx]),
        'seed_stability_ARI': stability
    })
metrics = pd.DataFrame(rows)
metrics.round(3)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,3.8))
axes[0].plot(metrics.k, metrics.inertia, marker='o'); axes[0].set_title('Inertia always falls')
axes[1].plot(metrics.k, metrics.silhouette, marker='o'); axes[1].set_title('Silhouette favors separation')
axes[2].plot(metrics.k, metrics.seed_stability_ARI, marker='o'); axes[2].set_title('Repeated starts test stability')
for ax in axes: ax.set_xlabel('number of clusters k')
plt.tight_layout(); plt.show()

### Three different validation questions

MIT CSAIL separates internal validation from external validation. For a business segmentation, add a third layer:

- **Internal geometry:** Are points cohesive within clusters and separated across clusters? Silhouette answers part of this question.
- **Stability:** Do plausible resamples, seeds, or feature choices return similar assignments?
- **External usefulness:** Do the groups predict a later outcome or support a treatment that creates incremental value?

A strong internal score cannot substitute for the external test. Random data can still produce a partition.

In [ ]:
# MIT CSAIL suggests visually auditing a similarity matrix after ordering rows by cluster.
audit_idx = rng.choice(len(Z), 240, replace=False)
audit_labels = labels_by_k[4][audit_idx]
order = np.argsort(audit_labels)
D = pairwise_distances(Z[audit_idx][order])
plt.figure(figsize=(6,5))
plt.imshow(D, cmap='magma_r', aspect='auto')
plt.colorbar(label='standardized Euclidean distance')
plt.title('A useful partition shows darker within-cluster blocks')
plt.xlabel('customers ordered by cluster')
plt.ylabel('customers ordered by cluster')
plt.show()

**Decision prompt:** If `k=3` has the best silhouette but `k=4` separates a small high-return group that needs a distinct service policy, which result would you recommend? State the additional evidence you would need.

## 5. Profiles turn cluster IDs into hypotheses

Cluster numbers have no inherent meaning. Profile medians, sizes, overlap, and unusual observations before naming a segment. Use factual names tied to behavior.

In [ ]:
chosen_k = 4
final_kmeans = KMeans(n_clusters=chosen_k, n_init=50, random_state=RANDOM_STATE)
customers_work = customers.copy()
customers_work['cluster'] = final_kmeans.fit_predict(Z)
profiles = customers_work.groupby('cluster')[features].median().round(2)
profiles.insert(0, 'customers', customers_work.groupby('cluster').size())
profiles.insert(1, 'share_pct', (100*customers_work.groupby('cluster').size()/len(customers_work)).round(1))
profiles

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(Z)
plt.figure(figsize=(8,5))
plt.scatter(coords[:,0], coords[:,1], c=customers_work.cluster, cmap='tab10', s=12, alpha=.55)
plt.xlabel('PCA 1'); plt.ylabel('PCA 2')
plt.title('PCA is a projection for inspection, not proof of cluster truth')
plt.show()
print('variance shown by two axes:', round(pca.explained_variance_ratio_.sum(),3))

## 6. Different algorithms ask different geometric questions

- **K-means:** which centroid is nearest? Best suited to compact, roughly convex groups after sensible scaling.
- **Hierarchical clustering:** which observations or groups merge next? The dendrogram preserves nested structure, but the distance and linkage rule determine the tree.
- **DBSCAN:** which observations lie in dense connected regions? It can identify noise and non-convex shapes, but `eps` and `min_samples` define density.

In [ ]:
# Compare methods on the same PCA coordinates for teaching visibility.
# The production analysis should fit on the prepared feature space, not only this 2D projection.
sample = rng.choice(len(Z), 700, replace=False)
Zs = Z[sample]
Ps = coords[sample]
method_labels = {
    'K-means, k=4': KMeans(n_clusters=4, n_init=30, random_state=RANDOM_STATE).fit_predict(Zs),
    'Hierarchical, 4 groups': AgglomerativeClustering(n_clusters=4, linkage='ward').fit_predict(Zs),
    'DBSCAN': DBSCAN(eps=1.1, min_samples=12).fit_predict(Zs),
}
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, (name, lab) in zip(axes, method_labels.items()):
    ax.scatter(Ps[:,0], Ps[:,1], c=lab, cmap='tab10', s=14, alpha=.65)
    ax.set_title(name + f'\nlabels: {len(set(lab))}, noise: {(lab==-1).sum()}')
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
plt.tight_layout(); plt.show()

## 7. A clustering stress test

Remove monetary value and average basket value, refit `k=4`, and compare assignments. If the story changes substantially, the original segmentation was partly a spending segmentation. That may be appropriate, but it should be stated explicitly.

In [ ]:
behavior_only = ['recency_days','purchase_frequency','product_diversity','return_line_rate']
Z_behavior = StandardScaler().fit_transform(np.log1p(customers[behavior_only].clip(lower=0)))
behavior_labels = KMeans(n_clusters=4, n_init=50, random_state=RANDOM_STATE).fit_predict(Z_behavior)
agreement = adjusted_rand_score(customers_work.cluster, behavior_labels)
print('Adjusted Rand agreement with spending features removed:', round(agreement,3))
pd.crosstab(customers_work.cluster, behavior_labels, normalize='index').round(2)

## 8. Final segment memo

For each proposed segment, write five lines:

1. **Measured profile:** the behaviors that distinguish it.
2. **Proposed action:** one treatment that differs from the global policy.
3. **Evidence:** silhouette, stability, profile separation, and size.
4. **Caveat:** overlap, sensitivity, or a subgroup that should not be forced into the story.
5. **Experiment:** how the retailer would test whether the differentiated action creates value.

Do not name segments with personality claims such as *loyal* or *unengaged* unless the measured behavior actually supports that claim.

**Suggested classroom flow:** define similarity, do one assignment/update by hand, explain why the mean lowers squared-distance cost, compare local solutions, separate internal validity from external usefulness, inspect the ordered distance matrix, profile `k=4`, and finish with the spending-removal stress test.

## References and further reading

- [ISLP: An Introduction to Statistical Learning with Python](https://www.statlearning.com/) — unsupervised learning and clustering.
- [Stanford CS229 k-means notes](https://cs229.stanford.edu/notes_archive/cs229-notes7a.pdf) — assignment, centroid update, and distortion.
- [MIT 15.097 clustering notes](https://ocw.mit.edu/courses/15-097-prediction-machine-learning-and-statistics-spring-2012/resources/lecture-5-clustering/) — objective, mean update, monotone improvement, local solutions, and cluster validation.
- [MIT CSAIL 6.S079 clustering lecture](https://dsg.csail.mit.edu/6.S079-2022/lectures/6.S079_Lec09.pdf) — internal versus external validation, similarity-matrix audits, hierarchical clustering, and DBSCAN.
- [Stanford STATS305C clustering notes](https://web.stanford.edu/class/stats305c/lectures/Clustering.html) — scaling, multiple initializations, and choosing cluster count.
- [CMU 10-601 clustering notes](https://www.cs.cmu.edu/afs/cs/academic/class/10601-f10/lecture/lec14.pdf) — distance, hierarchy, k-means, and method comparison.
- [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html) — algorithm assumptions and DBSCAN parameters.
- [UCI Online Retail dataset](https://doi.org/10.24432/C5BW33) — source for the course-prepared customer profiles.